In [1]:
%pip install pandas numpy scikit-learn plotly openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print("All libraries imported successfully.")

All libraries imported successfully.


In [3]:
BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Working directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

Working directory: C:\Users\Avishek\OneDrive\Desktop\Growfinix-Internship-Tasks\Task-3-Customer-Segmentation
Data directory: C:\Users\Avishek\OneDrive\Desktop\Growfinix-Internship-Tasks\Task-3-Customer-Segmentation\data
Output directory: C:\Users\Avishek\OneDrive\Desktop\Growfinix-Internship-Tasks\Task-3-Customer-Segmentation\outputs


In [4]:
FILE_NAME = "Online Retail.xlsx"

file_path = DATA_DIR / FILE_NAME

if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {file_path}"
    )

df = pd.read_excel(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
required_columns = [
    "InvoiceNo",
    "StockCode",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing columns: "
        + ", ".join(missing_columns)
    )

print("Dataset structure validated successfully.")

Dataset structure validated successfully.


In [6]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

df["Quantity"] = pd.to_numeric(
    df["Quantity"],
    errors="coerce"
)

df["UnitPrice"] = pd.to_numeric(
    df["UnitPrice"],
    errors="coerce"
)

df = df.dropna(
    subset=[
        "CustomerID",
        "InvoiceDate",
        "Quantity",
        "UnitPrice"
    ]
)

df["InvoiceNo"] = df["InvoiceNo"].astype(str)

df = df[
    ~df["InvoiceNo"]
    .str.startswith("C")
]

df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

print("Cleaned dataset shape:", df.shape)

Cleaned dataset shape: (397884, 8)


In [7]:
df["Revenue"] = (
    df["Quantity"] *
    df["UnitPrice"]
)

display(
    df[
        [
            "InvoiceNo",
            "CustomerID",
            "Quantity",
            "UnitPrice",
            "Revenue"
        ]
    ].head()
)

,InvoiceNo,CustomerID,Quantity,UnitPrice,Revenue
0,536365,17850.0,6,2.55,15.30
1,536365,17850.0,6,3.39,20.34
2,536365,17850.0,8,2.75,22.00
3,536365,17850.0,6,3.39,20.34
4,536365,17850.0,6,3.39,20.34


In [8]:
print("Total transactions:", len(df))

print(
    "Unique customers:",
    df["CustomerID"].nunique()
)

print(
    "Unique invoices:",
    df["InvoiceNo"].nunique()
)

print(
    "Unique products:",
    df["StockCode"].nunique()
)

print(
    "Total revenue:",
    round(df["Revenue"].sum(), 2)
)

Total transactions: 397884
Unique customers: 4338
Unique invoices: 18532
Unique products: 3665
Total revenue: 8911407.9


In [9]:
analysis_date = (
    df["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

In [10]:
customer_features = (
    df.groupby("CustomerID")
      .agg(
          Last_Purchase=(
              "InvoiceDate",
              "max"
          ),

          Frequency=(
              "InvoiceNo",
              "nunique"
          ),

          Monetary=(
              "Revenue",
              "sum"
          ),

          Total_Units=(
              "Quantity",
              "sum"
          ),

          Unique_Products=(
              "StockCode",
              "nunique"
          )
      )
      .reset_index()
)

In [11]:
customer_features["Recency"] = (
    analysis_date
    - customer_features["Last_Purchase"]
).dt.days

In [12]:
customer_features["Average_Order_Value"] = (
    customer_features["Monetary"]
    /
    customer_features["Frequency"]
)

In [13]:
customer_features["Average_Quantity_Per_Order"] = (
    customer_features["Total_Units"]
    /
    customer_features["Frequency"]
)

In [14]:
display(
    customer_features.head(10)
)

,CustomerID,Last_Purchase,Frequency,Monetary,Total_Units,Unique_Products,Recency,Average_Order_Value,Average_Quantity_Per_Order
0,12346.0,2011-01-18 10:01:00,1,77183.60,74215,1,326,77183.600000,74215.000000
1,12347.0,2011-12-07 15:52:00,7,4310.00,2458,103,2,615.714286,351.142857
2,12348.0,2011-09-25 13:13:00,4,1797.24,2341,22,75,449.310000,585.250000
3,12349.0,2011-11-21 09:51:00,1,1757.55,631,73,19,1757.550000,631.000000
4,12350.0,2011-02-02 16:01:00,1,334.40,197,17,310,334.400000,197.000000
5,12352.0,2011-11-03 14:37:00,8,2506.04,536,59,36,313.255000,67.000000
6,12353.0,2011-05-19 17:47:00,1,89.00,20,4,204,89.000000,20.000000
7,12354.0,2011-04-21 13:11:00,1,1079.40,530,58,232,1079.400000,530.000000
8,12355.0,2011-05-09 13:49:00,1,459.40,240,13,214,459.400000,240.000000
9,12356.0,2011-11-17 08:40:00,3,2811.43,1591,53,23,937.143333,530.333333


In [15]:
features = [
    "Recency",
    "Frequency",
    "Monetary",
    "Total_Units",
    "Unique_Products",
    "Average_Order_Value",
    "Average_Quantity_Per_Order"
]

X = customer_features[features].copy()

display(X.head())

,Recency,Frequency,Monetary,Total_Units,Unique_Products,Average_Order_Value,Average_Quantity_Per_Order
0,326,1,77183.60,74215,1,77183.600000,74215.000000
1,2,7,4310.00,2458,103,615.714286,351.142857
2,75,4,1797.24,2341,22,449.310000,585.250000
3,19,1,1757.55,631,73,1757.550000,631.000000
4,310,1,334.40,197,17,334.400000,197.000000


In [16]:
log_features = [
    "Monetary",
    "Total_Units",
    "Unique_Products",
    "Average_Order_Value",
    "Average_Quantity_Per_Order"
]

for column in log_features:

    X[column] = np.log1p(
        X[column]
    )

In [17]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(
    "Scaled feature matrix:",
    X_scaled.shape
)

Scaled feature matrix: (4338, 7)


In [18]:
k_results = []

for k in range(2, 9):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(
        X_scaled
    )

    score = silhouette_score(
        X_scaled,
        labels
    )

    k_results.append({
        "K": k,
        "Inertia": model.inertia_,
        "Silhouette": score
    })

k_results = pd.DataFrame(
    k_results
)

display(k_results)

,K,Inertia,Silhouette
0,2,19497.785378,0.326203
1,3,16246.678632,0.250074
2,4,14281.001915,0.251777
3,5,12434.843388,0.257677
4,6,11124.013163,0.241111
5,7,10087.618249,0.232162
6,8,9296.254028,0.225532


In [19]:
fig = px.line(
    k_results,
    x="K",
    y="Inertia",
    markers=True,
    title="K-Means Elbow Method"
)

fig.show()

In [20]:
fig = px.line(
    k_results,
    x="K",
    y="Silhouette",
    markers=True,
    title="Silhouette Score"
)

fig.show()

In [21]:
best_k = int(
    k_results.loc[
        k_results["Silhouette"].idxmax(),
        "K"
    ]
)

print(
    "Best K:",
    best_k
)

Best K: 2


In [22]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

customer_features["KMeans_Cluster"] = (
    kmeans.fit_predict(
        X_scaled
    )
)

In [23]:
kmeans_score = silhouette_score(
    X_scaled,
    customer_features["KMeans_Cluster"]
)

print(
    "K-Means Silhouette Score:",
    round(kmeans_score, 4)
)

K-Means Silhouette Score: 0.3262


In [24]:
min_samples = 8

neighbors = NearestNeighbors(
    n_neighbors=min_samples
)

neighbors_fit = neighbors.fit(
    X_scaled
)

distances, indices = (
    neighbors_fit.kneighbors(
        X_scaled
    )
)

k_distances = np.sort(
    distances[:, -1]
)

In [25]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=np.arange(
            len(k_distances)
        ),
        y=k_distances,
        mode="lines",
        name="k-distance"
    )
)

fig.update_layout(
    title="DBSCAN k-Distance Plot",
    xaxis_title="Sorted Customers",
    yaxis_title="Distance"
)

fig.show()

In [26]:
dbscan_results = []

for eps in np.arange(
    0.40,
    1.50,
    0.05
):

    model = DBSCAN(
        eps=float(eps),
        min_samples=min_samples
    )

    labels = model.fit_predict(
        X_scaled
    )

    clusters = (
        len(set(labels))
        - (1 if -1 in labels else 0)
    )

    noise = int(
        np.sum(labels == -1)
    )

    if clusters >= 2:

        mask = labels != -1

        score = silhouette_score(
            X_scaled[mask],
            labels[mask]
        )

    else:

        score = np.nan

    dbscan_results.append({
        "eps": round(
            float(eps),
            2
        ),
        "Clusters": clusters,
        "Noise": noise,
        "Silhouette": score
    })

dbscan_results = pd.DataFrame(
    dbscan_results
)

display(
    dbscan_results
)

,eps,Clusters,Noise,Silhouette
0,0.40,20,2463,-0.133709
1,0.45,7,1911,-0.110719
2,0.50,6,1488,-0.009052
3,0.55,5,1144,-0.027540
4,0.60,3,901,0.192821
5,0.65,1,718,NaN
6,0.70,3,578,0.139825
7,0.75,2,483,0.466541
8,0.80,2,412,0.464081
9,0.85,1,331,NaN


In [27]:
valid = dbscan_results[
    (dbscan_results["Clusters"] >= 2) &
    (dbscan_results["Noise"] <=
     len(customer_features) * 0.30)
].dropna(
    subset=["Silhouette"]
)

if len(valid) > 0:

    best_eps = float(
        valid.loc[
            valid["Silhouette"].idxmax(),
            "eps"
        ]
    )

else:

    best_eps = 0.75

print(
    "Selected eps:",
    best_eps
)

Selected eps: 0.75


In [28]:
dbscan = DBSCAN(
    eps=best_eps,
    min_samples=min_samples
)

customer_features["DBSCAN_Cluster"] = (
    dbscan.fit_predict(
        X_scaled
    )
)

In [29]:
db_labels = (
    customer_features[
        "DBSCAN_Cluster"
    ].values
)

db_clusters = (
    len(set(db_labels))
    -
    (1 if -1 in db_labels else 0)
)

db_noise = int(
    np.sum(
        db_labels == -1
    )
)

comparison = pd.DataFrame({
    "Model": [
        "K-Means",
        "DBSCAN"
    ],

    "Clusters": [
        best_k,
        db_clusters
    ],

    "Noise_Customers": [
        0,
        db_noise
    ],

    "Silhouette": [
        kmeans_score,
        np.nan
    ]
})

display(comparison)

,Model,Clusters,Noise_Customers,Silhouette
0,K-Means,2,0,0.326203
1,DBSCAN,2,483,NaN


In [30]:
pca = PCA(
    n_components=3,
    random_state=42
)

X_pca = pca.fit_transform(
    X_scaled
)

pca_df = pd.DataFrame(
    X_pca,
    columns=[
        "PC1",
        "PC2",
        "PC3"
    ]
)

pca_df["Cluster"] = (
    customer_features[
        "KMeans_Cluster"
    ].astype(str)
)

In [31]:
print(
    "PC1:",
    round(
        pca.explained_variance_ratio_[0] * 100,
        2
    ),
    "%"
)

print(
    "PC2:",
    round(
        pca.explained_variance_ratio_[1] * 100,
        2
    ),
    "%"
)

print(
    "PC3:",
    round(
        pca.explained_variance_ratio_[2] * 100,
        2
    ),
    "%"
)

print(
    "Total:",
    round(
        pca.explained_variance_ratio_.sum() * 100,
        2
    ),
    "%"
)

PC1: 58.77 %
PC2: 17.7 %
PC3: 10.84 %
Total: 87.32 %


In [32]:
fig = px.scatter_3d(
    pca_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Cluster",
    title="Customer Segmentation — 3D K-Means"
)

fig.update_traces(
    marker=dict(
        size=5,
        opacity=0.75
    )
)

fig.show()

In [33]:
segment_profile = (
    customer_features
    .groupby("KMeans_Cluster")
    [
        [
            "Recency",
            "Frequency",
            "Monetary",
            "Total_Units",
            "Unique_Products",
            "Average_Order_Value",
            "Average_Quantity_Per_Order"
        ]
    ]
    .mean()
    .round(2)
)

display(segment_profile)

,Recency,Frequency,Monetary,Total_Units,Unique_Products,Average_Order_Value,Average_Quantity_Per_Order
KMeans_Cluster,,,,,,,
0,48.62,6.56,3604.51,2107.64,97.56,604.18,377.28
1,141.15,1.74,338.38,177.03,21.59,214.39,116.45


In [34]:
overall = segment_profile.mean()

def classify_segment(row):

    if (
        row["Monetary"] > overall["Monetary"]
        and
        row["Frequency"] > overall["Frequency"]
        and
        row["Recency"] < overall["Recency"]
    ):
        return "High-Value Loyal"

    elif (
        row["Recency"] >
        overall["Recency"] * 1.35
    ):
        return "At-Risk / Inactive"

    elif (
        row["Frequency"] >
        overall["Frequency"]
        and
        row["Monetary"] <
        overall["Monetary"]
    ):
        return "Frequent Low-Spend"

    elif (
        row["Recency"] <
        overall["Recency"]
        and
        row["Frequency"] <
        overall["Frequency"]
    ):
        return "New / Promising"

    else:
        return "Regular / Growth Potential"

In [35]:
segment_names = {}

for cluster_id in segment_profile.index:

    segment_names[cluster_id] = (
        classify_segment(
            segment_profile.loc[
                cluster_id
            ]
        )
    )

customer_features[
    "Segment_Name"
] = customer_features[
    "KMeans_Cluster"
].map(
    segment_names
)

display(
    customer_features[
        [
            "CustomerID",
            "KMeans_Cluster",
            "Segment_Name"
        ]
    ].head(20)
)

,CustomerID,KMeans_Cluster,Segment_Name
0,12346.0,0,High-Value Loyal
1,12347.0,0,High-Value Loyal
2,12348.0,0,High-Value Loyal
3,12349.0,0,High-Value Loyal
4,12350.0,1,At-Risk / Inactive
5,12352.0,0,High-Value Loyal
6,12353.0,1,At-Risk / Inactive
7,12354.0,0,High-Value Loyal
8,12355.0,1,At-Risk / Inactive
9,12356.0,0,High-Value Loyal


In [36]:
marketing_strategy = {
    "High-Value Loyal":
        "VIP loyalty, premium products, early access, upselling and cross-selling",

    "At-Risk / Inactive":
        "Win-back emails, special offers, reminders and reactivation campaigns",

    "Frequent Low-Spend":
        "Bundle offers, volume discounts and cross-selling",

    "New / Promising":
        "Onboarding, education and second-purchase incentives",

    "Regular / Growth Potential":
        "Personalized recommendations and loyalty campaigns"
}

In [37]:
strategy_rows = []

for segment in sorted(
    customer_features[
        "Segment_Name"
    ].dropna().unique()
):

    count = int(
        (
            customer_features[
                "Segment_Name"
            ] == segment
        ).sum()
    )

    percentage = round(
        count /
        len(customer_features) *
        100,
        2
    )

    strategy_rows.append({

        "Segment":
            segment,

        "Customers":
            count,

        "Share_%":
            percentage,

        "Marketing_Strategy":
            marketing_strategy.get(
                segment,
                "Personalized campaign"
            )
    })

strategy_df = pd.DataFrame(
    strategy_rows
)

display(strategy_df)

,Segment,Customers,Share_%,Marketing_Strategy
0,At-Risk / Inactive,2059,47.46,"Win-back emails, special offers, reminders and..."
1,High-Value Loyal,2279,52.54,"VIP loyalty, premium products, early access, u..."


In [38]:
customer_features.to_csv(
    OUTPUT_DIR /
    "customer_features_and_segments.csv",
    index=False
)

segment_profile.to_csv(
    OUTPUT_DIR /
    "segment_profile.csv"
)

strategy_df.to_csv(
    OUTPUT_DIR /
    "marketing_strategy.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR /
    "model_comparison.csv",
    index=False
)

print("All outputs saved successfully.")

All outputs saved successfully.
